![cyber_photo](cyber_photo.jpg)

Cyber threats are a growing concern for organizations worldwide. These threats take many forms, including malware, phishing, and denial-of-service (DOS) attacks, compromising sensitive information and disrupting operations. The increasing sophistication and frequency of these attacks make it imperative for organizations to adopt advanced security measures. Traditional threat detection methods often fall short due to their inability to adapt to new and evolving threats. This is where deep learning models come into play.

Deep learning models can analyze vast amounts of data and identify patterns that may not be immediately obvious to human analysts. By leveraging these models, organizations can proactively detect and mitigate cyber threats, safeguarding their sensitive information and ensuring operational continuity.

As a cybersecurity analyst, you identify and mitigate these threats. In this project, you will design and implement a deep learning model to detect cyber threats. The BETH dataset simulates real-world logs, providing a rich source of information for training and testing your model. The data has already undergone preprocessing, and we have a target label, `sus_label`, indicating whether an event is malicious (1) or benign (0).

By successfully developing this model, you will contribute to enhancing cybersecurity measures and protecting organizations from potentially devastating cyber attacks.


### The Data

| Column     | Description              |
|------------|--------------------------|
|`processId`|The unique identifier for the process that generated the event - int64 |
|`threadId`|ID for the thread spawning the log - int64|
|`parentProcessId`|Label for the process spawning this log - int64|
|`userId`|ID of user spawning the log|Numerical - int64|
|`mountNamespace`|Mounting restrictions the process log works within - int64|
|`argsNum`|Number of arguments passed to the event - int64|
|`returnValue`|Value returned from the event log (usually 0) - int64|
|`sus_label`|Binary label as suspicous event (1 is suspicious, 0 is not) - int64|

More information on the dataset: [BETH dataset](accreditation.md)

In [44]:
# Import required libraries
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as functional
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from torchmetrics import Accuracy
# from sklearn.metrics import accuracy_score  # uncomment to use sklearn

In [45]:
# Load preprocessed data
train_df = pd.read_csv('labelled_train.csv')
test_df = pd.read_csv('labelled_test.csv')
val_df = pd.read_csv('labelled_validation.csv')

# View the first 5 rows of training set
train_df.head()

,processId,threadId,parentProcessId,userId,mountNamespace,argsNum,returnValue,sus_label
0,381,7337,1,100,4026532231,5,0,1
1,381,7337,1,100,4026532231,1,0,1
2,381,7337,1,100,4026532231,0,0,1
3,7347,7347,7341,0,4026531840,2,-2,1
4,7347,7347,7341,0,4026531840,4,0,1


In [47]:
features_train= train_df.drop(columns='sus_label').values
features_test= test_df.drop(columns='sus_label').values
features_valid = val_df.drop(columns='sus_label').values

label_train = train_df['sus_label']
label_test= test_df['sus_label']
label_valid = val_df['sus_label']

scaler = StandardScaler()

scaled_feature_train  = scaler.fit_transform(features_train)
scaled_feature_test = scaler.transform(features_test)
scaled_feature_valid = scaler.transform(features_valid)

feature_train_tensor = torch.tensor(scaled_feature_train, dtype=torch.float32)
feature_test_tensor = torch.tensor(scaled_feature_test, dtype=torch.float32)
feature_valid_tensor = torch.tensor(scaled_feature_valid, dtype=torch.float32)

label_train_tensor = torch.tensor(label_train,dtype=torch.float32).view(-1,1)
label_test_tensor = torch.tensor(label_test,dtype=torch.float32).view(-1,1)
label_valid_tensor = torch.tensor(label_valid,dtype=torch.float32).view(-1,1)

model = nn.Sequential(
    nn.Linear(feature_train_tensor.shape[1], 128),  
 nn.LeakyReLU( 
negative_slope = 0.05) , 
    nn.Linear(128, 64),  
    nn.LeakyReLU( 
negative_slope = 0.05) ,  
    nn.Linear(64, 1)
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, weight_decay=0.0001)

for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    pred  = model(feature_train_tensor)
    loss = criterion(pred,label_train_tensor)
    loss.backward()
    optimizer.step()


model.eval()
with torch.no_grad():
    label_pred_train = model(feature_train_tensor).round()
    label_pred_test = model(feature_test_tensor).round()
    label_pred_valid = model(feature_valid_tensor).round()


accuracy = Accuracy(task='binary')

acc_train = accuracy(label_pred_train,label_train_tensor)
acc_test = accuracy(label_pred_test,label_test_tensor)
acc_valid = accuracy(label_pred_valid,label_valid_tensor)

train_accuracy = acc_train.item()
test_accuracy = acc_test.item()
val_accuracy = acc_valid.item()

print(f'Training accuracy : {train_accuracy}')
print(f'Test accuracy : {test_accuracy}')
print(f'Validation accuracy : {val_accuracy}')



Training accuracy : 0.9988036155700684
Test accuracy : 0.9439266920089722
Validation accuracy : 0.9974122643470764
